In [13]:
import json, os
import cv2
import numpy as np
from huggingface_hub import hf_hub_download

In [22]:
# 1. Setup paths and directories
json_path = '../Datasets/multi_face.json'
image_path = '../Datasets/multi_face.jpg'          # Change to your target image file
output_dir = '../Datasets/extracted_faces'       # Folder where cropped faces will be saved
os.makedirs(output_dir, exist_ok=True)
output_json_path = '../Datasets/detected_faces.json' # File where face coordinates will be saved

# ADJUST PADDING HERE: 0.2 means adding 20% of the face width/height as a margin around all sides
padding_percentage = 0.2 

In [15]:
# 2. Correctly open and load the JSON file
with open(json_path, 'r') as file:
    detections = json.load(file)

In [16]:
# 3. Load the source image
image = cv2.imread(image_path)
if image is None:
    raise FileNotFoundError(f"Could not load image at {image_path}")
h_img, w_img, _ = image.shape

In [17]:
# 4. Initialize YuNet
model_path = hf_hub_download(repo_id="opencv/face_detection_yunet", filename="face_detection_yunet_2023mar.onnx")
yunet = cv2.FaceDetectorYN.create(
    model=model_path,
    config="",
    input_size=(320, 320), # Temporary placeholder size
    score_threshold=0.6,
    nms_threshold=0.3
)

In [24]:
face_counter = 0
yunet_results = []  # List to store JSON metadata records

# 5. Process each region from the initial JSON data
for item in detections:
    box = item.get("box") or item.get("bbox")
    if not box:
        continue
        
    # Assuming pixel coordinates: [xmin, ymin, xmax, ymax]
    xmin, ymin, xmax, ymax = map(int, box)
    
    # Clip coordinates to prevent crashing at image boundaries
    xmin, ymin = max(0, xmin), max(0, ymin)
    xmax, ymax = min(w_img, xmax), min(h_img, ymax)
    
    # Slice out the initial crop region
    roi = image[ymin:ymax, xmin:xmax]
    if roi.size == 0:
        continue
        
    # Scale YuNet dynamically to match this specific crop's shape
    roi_h, roi_w, _ = roi.shape
    yunet.setInputSize((roi_w, roi_h))
    
    # Run face detection inside the ROI
    _, faces = yunet.detect(roi)
    
    # 6. Extract, Pad, Save Images, and Record JSON Metadata
    if faces is not None:
        for face in faces:
            # Extract localized bounding box coordinates and confidence from YuNet
            fx, fy, fw, fh = map(int, face[:4])
            confidence = float(face[-1]) # YuNet confidence score is the last value
            
            # Map coordinates back to the global image coordinate map
            global_xmin = xmin + fx
            global_ymin = ymin + fy
            global_xmax = global_xmin + fw
            global_ymax = global_ymin + fh
            
            # --- PADDING CALCULATION ---
            pad_w = int(fw * padding_percentage)
            pad_h = int(fh * padding_percentage)
            
            # Apply padding expand margins outwards
            padded_xmin = max(0, global_xmin - pad_w)
            padded_ymin = max(0, global_ymin - pad_h)
            padded_xmax = min(w_img, global_xmax + pad_w)
            padded_ymax = min(h_img, global_ymax + pad_h)
            # ---------------------------
            
            # Crop out the final padded face array
            face_crop = image[padded_ymin:padded_ymax, padded_xmin:padded_xmax]
            
            if face_crop.size > 0:
                face_counter += 1
                face_filename = os.path.join(output_dir, f"padded_face_{face_counter}.jpg")
                
                # Save image file to disk
                cv2.imwrite(face_filename, face_crop)
                
                # Append standard metadata format to output list
                # Uses global global coordinates before or after padding based on preference (Padded recorded below)
                yunet_results.append({
                    "face_box": [padded_xmin, padded_ymin, padded_xmax, padded_ymax],
                    "confidence": round(confidence, 2)
                })
                
                print(f"Saved: {face_filename} | Confidence: {round(confidence, 2)}")

# 7. Write accumulated face tracking dictionary array out to JSON file
with open(output_json_path, 'w') as out_json:
    json.dump(yunet_results, out_json, indent=2)

print(f"\nExtraction complete!")
print(f"Total faces saved: {face_counter}")
print(f"Coordinates database written to: {output_json_path}")

Saved: ../Datasets/extracted_faces/padded_face_1.jpg | Confidence: 0.91
Saved: ../Datasets/extracted_faces/padded_face_2.jpg | Confidence: 0.92
Saved: ../Datasets/extracted_faces/padded_face_3.jpg | Confidence: 0.91
Saved: ../Datasets/extracted_faces/padded_face_4.jpg | Confidence: 0.91
Saved: ../Datasets/extracted_faces/padded_face_5.jpg | Confidence: 0.89

Extraction complete!
Total faces saved: 5
Coordinates database written to: ../Datasets/detected_faces.json
